# Machine Learning Experiment for Crashes Near Schools

First lets import the important libaries like
os
pandas
numpy
matplotlib

The list will update as move on this experiment

In [13]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Read the data which was scrapped from crash_data, walking network, traffic network, cycling network and others.

In [14]:
df = pd.read_csv("outputs/ml_features.csv")
#View the first 10 entries of the data
df.head(10)

,ACCIDENT_NO,nearest_school,serious_or_fatal,hour,day_of_week,month,is_weekend,is_school_hours,speed_zone_num,is_high_speed_zone,...,arterial_count_800m,arterial_pct_800m,avg_speed_800m,high_speed_road_800m,crossings_800m,signals_800m,crossing_density_800m,cycle_length_800m,protected_cycle_length_800m,cycle_pct_800m
0,T20230023803,Preston HS,1,8,3,10,0,1,80,1,...,202,32.53,46.3,106,425,27,1.83,108845.3,4586.2,46.95
1,T20240010211,Preston HS,0,17,1,4,0,1,50,0,...,202,32.53,46.3,106,425,27,1.83,108845.3,4586.2,46.95
2,T20240028305,William Ruthven SC,0,11,3,11,0,0,40,0,...,67,12.71,65.2,58,201,13,1.37,58638.3,1477.6,40.11
3,T20210020724,Reservoir HS,1,19,3,8,0,0,80,1,...,91,24.07,57.7,55,192,23,1.25,67218.0,6162.9,43.86
4,T20240031629,Preston HS,1,8,1,12,0,1,60,1,...,202,32.53,46.3,106,425,27,1.83,108845.3,4586.2,46.95
5,T20230006766,William Ruthven SC,0,5,5,3,1,0,100,1,...,67,12.71,65.2,58,201,13,1.37,58638.3,1477.6,40.11
6,T20230001177,William Ruthven SC,0,19,6,12,1,0,999,1,...,67,12.71,65.2,58,201,13,1.37,58638.3,1477.6,40.11
7,T20240008949,Preston HS,1,9,2,4,0,1,60,1,...,202,32.53,46.3,106,425,27,1.83,108845.3,4586.2,46.95
8,T20250018137,William Ruthven SC,0,10,5,7,1,0,999,1,...,67,12.71,65.2,58,201,13,1.37,58638.3,1477.6,40.11
9,T20250018069,Preston HS,1,14,0,7,0,1,999,1,...,202,32.53,46.3,106,425,27,1.83,108845.3,4586.2,46.95


Sucessfully the data set is imported to study. At, first need to do some data wrangling process.

Moving forward with Understanding the data.

### Understanding the data

In [15]:
# Check the dimmensions
print(df.shape)

(7773, 62)


So, the data have 7773 rows and 62 columns

In [16]:
# Check Column Names
print(df.columns)

Index(['ACCIDENT_NO', 'nearest_school', 'serious_or_fatal', 'hour',
       'day_of_week', 'month', 'is_weekend', 'is_school_hours',
       'speed_zone_num', 'is_high_speed_zone', 'road_geometry_code',
       'no_of_vehicles', 'light_condition_code', 'is_dark', 'dist_to_gate_m',
       'near_school_400m', 'cys_score', 'walk_edges_200m', 'walk_length_200m',
       'footpath_length_200m', 'footpath_pct_200m', 'road_count_200m',
       'arterial_count_200m', 'arterial_pct_200m', 'avg_speed_200m',
       'high_speed_road_200m', 'crossings_200m', 'signals_200m',
       'crossing_density_200m', 'cycle_length_200m',
       'protected_cycle_length_200m', 'cycle_pct_200m', 'walk_edges_400m',
       'walk_length_400m', 'footpath_length_400m', 'footpath_pct_400m',
       'road_count_400m', 'arterial_count_400m', 'arterial_pct_400m',
       'avg_speed_400m', 'high_speed_road_400m', 'crossings_400m',
       'signals_400m', 'crossing_density_400m', 'cycle_length_400m',
       'protected_cycle_length_

From this we have `'serious_or_fatal'` column as target variable and `'ACCIDENT_NO'` is an identifier.

In [17]:
# Check data tyoes and missing values
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7773 entries, 0 to 7772
Data columns (total 62 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ACCIDENT_NO                  7773 non-null   object 
 1   nearest_school               7773 non-null   object 
 2   serious_or_fatal             7773 non-null   int64  
 3   hour                         7773 non-null   int64  
 4   day_of_week                  7773 non-null   int64  
 5   month                        7773 non-null   int64  
 6   is_weekend                   7773 non-null   int64  
 7   is_school_hours              7773 non-null   int64  
 8   speed_zone_num               7773 non-null   int64  
 9   is_high_speed_zone           7773 non-null   int64  
 10  road_geometry_code           7773 non-null   int64  
 11  no_of_vehicles               7773 non-null   int64  
 12  light_condition_code         7773 non-null   int64  
 13  is_dark           

Form the above we found that, 
* cys_score is completly null (it was data scrapping problem Which is fixed in feature_engineering.py)
* avg_speed_200m and high_speed_200m got some missing values
* signals_200m got major missing values.

In [18]:
# Statistical Summary
print(df.describe())

       serious_or_fatal         hour  day_of_week        month   is_weekend  \
count       7773.000000  7773.000000  7773.000000  7773.000000  7773.000000   
mean           0.431108    13.321112     3.117844     6.036279     0.307989   
std            0.495263     4.717120     1.977422     3.400482     0.461692   
min            0.000000     0.000000     0.000000     1.000000     0.000000   
25%            0.000000    10.000000     1.000000     3.000000     0.000000   
50%            0.000000    14.000000     3.000000     6.000000     0.000000   
75%            1.000000    17.000000     5.000000     9.000000     1.000000   
max            1.000000    23.000000     6.000000    12.000000     1.000000   

       is_school_hours  speed_zone_num  is_high_speed_zone  \
count      7773.000000     7773.000000         7773.000000   
mean          0.354947      184.446288            0.639264   
std           0.478528      314.225500            0.480245   
min           0.000000       30.000000  

From the above we couldnt find any outliers but there might be which we will proceed to check on later stage.

### Remove Duplicates

In [19]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
7768    False
7769    False
7770    False
7771    False
7772    False
Length: 7773, dtype: bool

From the above we cant see any duplicates for first and last rows.
Lets sum up for any duplicates.

In [20]:
df.duplicated().sum()

0

So, there is no duplicate entries we can just continue with other steps.

### Handling Null Values

Since, from the above we have found some null values in some colunns which we need to dealw with it.

First lets check the sum of null value

In [21]:
print(df.isnull().sum())

ACCIDENT_NO                    0
nearest_school                 0
serious_or_fatal               0
hour                           0
day_of_week                    0
                              ..
signals_800m                   0
crossing_density_800m          0
cycle_length_800m              0
protected_cycle_length_800m    0
cycle_pct_800m                 0
Length: 62, dtype: int64
